In [ ]:
"""
CNC-DEC Hyperparameter Tuning -- SKELETON, adapted from Brayan's
xgboost_BayesianHyperparameterTuning.ipynb protocol (Bayesian Hyperparameter
Tuning/utils/train_evaluate.py :: bayesian_optimization()).

THIS IS A STARTING STRUCTURE, NOT A FINISHED SCRIPT. It mirrors the exact
pattern used for XGBoost tuning in this repo -- same nested bootstrap x
Bayesian-search shape, same bayes_opt library, same stratified split ratios --
adapted to CNC-DEC's actual CLI (see CNC-DEC/run_cncdec.py) and its
clustering-based hyperparameter surface. Fill in the TODOs before running,
and time ONE bootstrap iteration first (see bottom of file) before committing
to the full 100-bootstrap loop -- per Prof. Barman's timing gate from the
7/27 meeting.

Key difference from the XGBoost version: CNC-DEC is unsupervised (DEC
clustering), so there's no validation "AUROC" to optimize against during
each Bayesian trial. The label is only used for REPORTING afterward
(run_cncdec.py docstring, line 77-78). For hyperparameter *tuning* to make
sense, we need a numeric target to maximize -- the natural choice is
validation-set clustering agreement (ARI or accuracy) against the known
MGS label, computed the same way run_cncdec.py already does via
models.metrics.calculate_metrics. This uses the label only to SCORE
candidate hyperparameters, not to train the model -- worth explicitly
flagging this framing to Prof. Barman, since it's a design choice, not
something dictated by the existing code.
"""

import os
import sys
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from bayes_opt import BayesianOptimization, UtilityFunction

# CNC-DEC/ needs to be on the path -- adjust to wherever you cloned the repo
sys.path.append("RNAseq-AMD/CNC-DEC")
from misc.dataset import get_data, DEFAULT_CLIN_FILE, DEFAULT_RNA_FILE       # noqa: E402
from misc.helpers import normalizeRNA                                        # noqa: E402
from models.cncvae import CNCVAE                                             # noqa: E402
from models.dec import DEC                                                   # noqa: E402
from models.metrics import calculate_metrics                                 # noqa: E402

rng = np.random.default_rng(2026)

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
N_BOOTSTRAP = 100          # outer loop -- matches Brayan's protocol
N_BAYES_ITER = 100         # inner Bayesian search trials per bootstrap
N_INIT_POINTS = 20         # initial random exploration points (matches XGBoost script)
N_CLUSTERS = 2             # 2 for MGS1 vs MGS4; adjust for pairwise/4-class runs
LABEL_COL = "mgs_level"

# TODO: point these at your reduced feature set (81 genes + your 17 metadata
# features) rather than the full clinical file -- run_cncdec.py's
# --clinical_mode curated may already be close to what you want; compare
# it against your 17 before deciding.
RNA_FILE = DEFAULT_RNA_FILE
CLIN_FILE = DEFAULT_CLIN_FILE


# Bayesian search space -- CNC-VAE / DEC hyperparameters from run_cncdec.py's
# argparse definitions. Bounds below are starting guesses, NOT validated --
# review with Prof. Barman before trusting them.
PARAM_BOUNDS = {
    "ds": (16, 128),          # encoder/decoder hidden layer size
    "ls": (4, 16),            # latent dimension (ties to the 4/8/16 sweep already run)
    "dropout": (0.0, 0.5),
    "beta": (0.1, 5.0),       # beta-VAE KL weight
    "epochs": (50, 300),      # CNC-VAE pretraining epochs
    "bs": (8, 64),            # CNC-VAE pretraining batch size
    "dec_update_interval": (10, 100),
    "dec_tol": (0.001, 0.05),
}
# NOTE: 'act' (activation) and 'distance' (kl/mmd) are categorical, not
# continuous -- bayes_opt only searches continuous ranges. Handle these the
# same way run_cncdec.py's own random_forest branch handles max_features:
# search a continuous [0, N) range and floor/int() it to an index into a
# fixed list, e.g. ['elu', 'relu', 'tanh'][int(act_choice)].


def cncdec_objective_factory(x_num_train, x_bin_train, y_train,
                              x_num_val, x_bin_val, y_val, seed):
    """Returns a function bayes_opt can call: takes hyperparameters, trains
    CNC-DEC on the train split, scores clustering agreement (ARI) on the
    held-out validation split. This is the direct analog of
    xgboost_hyperparam() in Brayan's train_evaluate.py (lines 137-174)."""

    def cncdec_hyperparam(ds, ls, dropout, beta, epochs, bs,
                           dec_update_interval, dec_tol):
        ds, ls = int(ds), int(ls)
        epochs, bs = int(epochs), int(bs)
        dec_update_interval = int(dec_update_interval)

        input_size = x_num_train.shape[1] + x_bin_train.shape[1]
        autoencoder = CNCVAE(
            input_size=input_size, ds=ds, ls=ls, act="elu", dropout=dropout,
            distance="kl", beta=beta, epochs=epochs, bs=bs,
        )
        autoencoder.build_model(seed=seed)
        autoencoder.train(x_num_train, x_bin_train, seed=seed)

        dec = DEC(autoencoder, n_clusters=N_CLUSTERS, seed=seed)
        # TODO: confirm DEC.fit can be pointed at a train split and scored
        # on a separate val split -- run_cncdec.py trains/predicts on the
        # same full dataset (it's fully unsupervised), so this may need
        # DEC's fit() and a separate predict-only pass on x_num_val/x_bin_val.
        # Check models/dec.py's public methods before assuming this works
        # as written.
        y_pred, y_proba, centroids, z = dec.fit(
            x_num_val, x_bin_val, y=y_val,
            maxiter=2000, batch_size=64,
            update_interval=dec_update_interval, tol=dec_tol, n_init=20,
        )

        acc, ari, nmi = calculate_metrics(y_val, y_pred)
        return ari  # maximize validation ARI; swap for acc/nmi if preferred

    return cncdec_hyperparam


def bayesian_tune_cncdec(x_num, x_bin, y, n_bootstrap=N_BOOTSTRAP,
                          n_bayes_iter=N_BAYES_ITER, seed_base=2026):
    """Outer bootstrap loop, mirrors bayesian_optimization() in
    Bayesian Hyperparameter Tuning/utils/train_evaluate.py lines 107-322."""
    results = []

    for i in range(n_bootstrap):
        seed = seed_base + i
        # Same split ratios as Brayan's protocol: 20% test, then 20% of the
        # remainder as val -> ~64% train / 16% val / 20% test, stratified.
        idx = np.arange(len(y))
        idx_trainval, idx_test = train_test_split(
            idx, stratify=y, test_size=0.2, random_state=rng.integers(500000)
        )
        idx_train, idx_val = train_test_split(
            idx_trainval, stratify=y[idx_trainval], test_size=0.2,
            random_state=rng.integers(500000)
        )

        objective = cncdec_objective_factory(
            x_num[idx_train], x_bin[idx_train], y[idx_train],
            x_num[idx_val], x_bin[idx_val], y[idx_val], seed,
        )

        optimizer = BayesianOptimization(
            f=objective, pbounds=PARAM_BOUNDS, random_state=1, verbose=0,
        )
        optimizer.set_gp_params(alpha=1e-5)
        utility = UtilityFunction(kind="poi", xi=0.0)
        optimizer.maximize(
            init_points=N_INIT_POINTS, n_iter=n_bayes_iter,
            acquisition_function=utility,
        )

        best = max(optimizer.res, key=lambda d: d["target"])
        # TODO: refit best hyperparams on train+val, evaluate on idx_test,
        # record test ARI/NMI/accuracy -- mirrors lines 249-321 of
        # train_evaluate.py (refit best model, score on held-out test).

        results.append({"bootstrap": i, "best_val_ari": best["target"],
                         **best["params"]})
        print(f"  bootstrap {i+1}/{n_bootstrap}: best val ARI = {best['target']:.4f}")

    return pd.DataFrame(results)


if __name__ == "__main__":
    # ---- STEP 1: TIME A SINGLE RUN FIRST (per Prof. Barman's timing gate) ----
    # Load data, then time just ONE bootstrap iteration with a small
    # n_bayes_iter before committing to the full 100 x 100 loop.
    data = get_data(RNA_FILE, CLIN_FILE, LABEL_COL, clinical_mode="curated")
    x_num = normalizeRNA(data["rnanp"])
    x_bin = data["clin"]
    y = data["y"]

    print("Timing a single bootstrap iteration (reduced n_bayes_iter=5)...")
    t0 = time.time()
    _ = bayesian_tune_cncdec(x_num, x_bin, y, n_bootstrap=1, n_bayes_iter=5)
    elapsed = time.time() - t0
    print(f"1 bootstrap x 5 Bayes trials took {elapsed:.1f}s")
    est_full = elapsed * (N_BOOTSTRAP / 1) * (N_BAYES_ITER / 5)
    print(f"Estimated full run (100 bootstrap x 100 trials): "
          f"{est_full/3600:.1f} hours -- decide whether to proceed per the "
          f"meeting's timing gate (proceed if ~30min-ish per bootstrap, "
          f"skip/reduce if this balloons toward ~4hrs+ per bootstrap).")

    # ---- STEP 2: full run, only after the timing check above looks reasonable ----
    # results_df = bayesian_tune_cncdec(x_num, x_bin, y)
    # results_df.to_csv("cncdec_bayesian_tuning_results.csv", index=False)